**An investigation into how to identify geolocation points outside of Virginia**

## Purpose
This notebook explores methodology for identifying wildlife collision records that fall outside Virginia's geographic boundaries using geospatial analysis.

## Methodology

### 1. Test Data Generation
- Created a synthetic dataset with 90 random points inside Virginia's approximate bounds (lat: 36.5-39.5°, lon: -83.7 to -75.0°)
- Added 10 known points outside Virginia (Los Angeles, NYC, Miami, Chicago, Austin, Tokyo, London, "Null Island", Sydney, Moscow)
- Total: 100 test points labeled as "inside" or "outside"

### 2. Geospatial Analysis Approach
- Loaded US state shapefile (Census Bureau's cb_2018_us_state_20m) containing Virginia boundaries
- Converted latitude/longitude coordinates to GeoPandas Point geometries
- Handled CRS (Coordinate Reference System) alignment:
  - Points initialized as WGS84 (EPSG:4326)
  - Virginia shapefile in NAD83 (EPSG:4269)
  - Reprojected points to match shapefile CRS

### 3. Spatial Join Operation
- Performed left spatial join using `within` predicate to test point containment
- Points outside Virginia identified by `NaN` values in `index_right` column (no match found)

## Key Findings
- Successfully identified points outside Virginia boundaries
- The methodology correctly flagged coordinates beyond Virginia's borders using geometric containment rather than simple bounding box checks

## Technical Notes
- Uses GeoPandas for geospatial operations
- Requires proper CRS alignment for accurate spatial analysis
- The `within` predicate provides precise boundary checking against actual state geometry

## Application
This approach can be applied to the full WRMD dataset to filter or flag records that fall outside Virginia's jurisdiction.


In [1]:
import pandas as pd
import numpy as np

# Virginia approximate bounds
va_lat_min, va_lat_max = 36.5, 39.5
va_lon_min, va_lon_max = -83.7, -75.0

# Generate points inside Virginia
n_inside = 90
inside_latitudes = np.random.uniform(va_lat_min, va_lat_max, n_inside)
inside_longitudes = np.random.uniform(va_lon_min, va_lon_max, n_inside)

# Outside points (e.g., California, Florida, or Atlantic Ocean)
outside_coords = [
    (34.0522, -118.2437),  # Los Angeles, CA
    (40.7128, -74.0060),   # NYC
    (25.7617, -80.1918),   # Miami, FL
    (41.8781, -87.6298),   # Chicago, IL
    (30.2672, -97.7431),   # Austin, TX
    (35.6895, 139.6917),   # Tokyo
    (51.5074, -0.1278),    # London
    (0.0, 0.0),            # Null Island
    (-33.8688, 151.2093),  # Sydney, AU
    (55.7558, 37.6173)     # Moscow
]

# Split into separate lists
outside_latitudes, outside_longitudes = zip(*outside_coords)

# Combine inside and outside data
latitudes = np.concatenate([inside_latitudes, outside_latitudes])
longitudes = np.concatenate([inside_longitudes, outside_longitudes])

# Labels: "inside" for the first 90, "outside" for the rest
labels = ['inside'] * n_inside + ['outside'] * len(outside_coords)

# Create DataFrame
df = pd.DataFrame({
    'latitude': latitudes,
    'longitude': longitudes,
    'location_type': labels
})

# Show first few rows
print(df.head())



    latitude  longitude location_type
0  37.509349 -77.442952        inside
1  36.508361 -75.486507        inside
2  37.938878 -76.799147        inside
3  38.022139 -79.536040        inside
4  39.280124 -82.258187        inside


In [3]:
# Libraries
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import os

# Load Virginia shapefile

va_shapefile = "../datasets/cb_2018_us_state_20m/cb_2018_us_state_20m.shp"
virginia = gpd.read_file(va_shapefile)

print("✅ Loaded Virginia shapefile")
print("Virginia CRS:", virginia.crs)

#  Load your latitude/longitude dataset 
csv_path = "path/to/your/coordinates.csv"
points_df = df

# Create geometry column from lon/lat
geometry = [Point(xy) for xy in zip(points_df.longitude, points_df.latitude)]
points_gdf = gpd.GeoDataFrame(points_df, geometry=geometry)

# Set CRS to WGS84 (EPSG:4326) if not already set
if points_gdf.crs is None:
    points_gdf.set_crs(epsg=4326, inplace=True)
    print("✅ Set CRS for points to EPSG:4326 (WGS84)")

print("Points CRS:", points_gdf.crs)

# Check and align CRS 
if points_gdf.crs != virginia.crs:
    print("⚠️ CRS mismatch detected. Reprojecting points to match Virginia shapefile...")
    points_gdf = points_gdf.to_crs(virginia.crs)
    print("Points reprojected. CRS is now:", points_gdf.crs)
else:
    print("CRS match. No reprojection needed.")

#  Spatial join to find which points are within Virginia
joined = gpd.sjoin(points_gdf, virginia, how="left", predicate='within')

# Points outside Virginia will have NaN in 'index_right' (no match found)
outside_va = joined[joined['index_right'].isna()]

# Result 
print(" Points OUTSIDE Virginia:")
print(outside_va[['longitude', 'latitude']].head())


✅ Loaded Virginia shapefile
Virginia CRS: EPSG:4269
✅ Set CRS for points to EPSG:4326 (WGS84)
Points CRS: EPSG:4326
⚠️ CRS mismatch detected. Reprojecting points to match Virginia shapefile...
Points reprojected. CRS is now: EPSG:4269
 Points OUTSIDE Virginia:
    longitude   latitude
1  -75.486507  36.508361
23 -76.383737  37.228536
25 -76.372218  38.884006
32 -75.546389  37.571601
53 -76.468473  38.669527


In [ ]:
virginia.buffer()

<bound method GeoPandasBase.buffer of    STATEFP   STATENS     AFFGEOID GEOID STUSPS                  NAME LSAD  \
0       24  01714934  0400000US24    24     MD              Maryland   00   
1       19  01779785  0400000US19    19     IA                  Iowa   00   
2       10  01779781  0400000US10    10     DE              Delaware   00   
3       39  01085497  0400000US39    39     OH                  Ohio   00   
4       42  01779798  0400000US42    42     PA          Pennsylvania   00   
5       31  01779792  0400000US31    31     NE              Nebraska   00   
6       53  01779804  0400000US53    53     WA            Washington   00   
7       72  01779808  0400000US72    72     PR           Puerto Rico   00   
8       01  01779775  0400000US01    01     AL               Alabama   00   
9       05  00068085  0400000US05    05     AR              Arkansas   00   
10      35  00897535  0400000US35    35     NM            New Mexico   00   
11      48  01779801  0400000US48    4